# 04 - System evaluation
Owner: Oussema | all 5 compare results here

Launch Jupyter **from the repo root** (see README). Requirements: FAISS index built, Neo4j running and
loaded, an LLM configured in `config/.env` (see `docs/setup.md`). Without them the cells still run:
failed questions are listed as errors instead of stopping the notebook.

Sections: 1. end-to-end evaluation, 2. failures and reviewer findings, 3. Classic vs GraphRAG vs Hybrid
retrieval, 4. vector-only vs hybrid end to end, 5. chart.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "agents").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from tests.eval.evaluate import evaluate, format_summary, load_questions

questions = load_questions()
print(f"{len(questions)} questions, categories:", sorted({q['category'] for q in questions}))

## 1. End-to-end evaluation (hybrid retrieval)

In [ ]:
rows_hybrid, summary_hybrid = evaluate(top_k=5, use_graph=True, tag="hybrid", verbose=False)
print(format_summary(summary_hybrid, title="Hybrid (FAISS + Neo4j)"))

Columns: **intent** = detected intent equals the expected one, **framework** = the expected framework
appears among the sources, **review** = answers accepted by the reviewer (`agents/evaluation_assistant.review`),
**cited** = answers citing their sources, **refusal** = out-of-scope questions answered with an honest
"not found", **err** = questions that crashed (e.g. a stage not implemented yet).

## 2. Failures and reviewer findings

In [ ]:
def show(title, selected):
    print(f"\n--- {title}: {len(selected)}")
    for r in selected:
        print(f"Q{r['id']:>2} [{r['category']}] {r['question'][:90]}")
        if r["error"]:
            print("     error  :", r["error"])
        else:
            print(f"     intent : {r['intent']} (expected {r['expected_intent']}) | sources: {r['frameworks']}")
            for issue in r["review_issues"]:
                print("     review :", issue)

show("errors", [r for r in rows_hybrid if r["error"]])
show("wrong intent", [r for r in rows_hybrid if not r["error"] and not r["intent_ok"]])
show("framework not retrieved", [r for r in rows_hybrid if r["framework_ok"] is False and not r["error"]])
show("rejected by the reviewer", [r for r in rows_hybrid if r["review_ok"] is False and not r["error"]])

## 3. Classic vs GraphRAG vs Hybrid retrieval
Retrieval only (no LLM): does the expected framework appear in the top-k, and how much do the
two retrievers overlap? Only questions with an `expected_framework` are counted.

In [ ]:
try:
    from retrieval.faiss_retriever import retrieve_vector as vector_search      # name in the skeleton
except ImportError:
    from retrieval.faiss_retriever import search_faiss as vector_search          # name in feat/retrieval
from retrieval.graph_retriever import retrieve_graph
from retrieval.hybrid_retriever import retrieve

K = 5
MODES = {
    "Classic (FAISS)": lambda q: vector_search(q, k=K),
    "GraphRAG (Neo4j)": lambda q: retrieve_graph(q, k=K),
    "Hybrid (RRF)": lambda q: retrieve(q, k=K, use_graph=True),
}
targets = [q for q in questions if q["expected_framework"]]
hits = {m: 0 for m in MODES}
returned = {m: 0 for m in MODES}
failed = {m: 0 for m in MODES}
ids = {m: {} for m in MODES}

for q in targets:
    for mode, search in MODES.items():
        try:
            chunks = search(q["question"])
        except Exception as e:                               # NotImplementedError, Neo4j down...
            failed[mode] += 1
            chunks = []
        ids[mode][q["id"]] = {c["chunk_id"] for c in chunks}
        returned[mode] += bool(chunks)
        hits[mode] += q["expected_framework"] in {c["framework"] for c in chunks}

print(f"{'mode':18} {'framework hit':>14} {'non-empty':>10} {'errors':>7}")
for mode in MODES:
    print(f"{mode:18} {hits[mode]:>9}/{len(targets):<4} {returned[mode]:>10} {failed[mode]:>7}")

both = [len(ids['Classic (FAISS)'][i] & ids['GraphRAG (Neo4j)'][i]) for i in ids['Classic (FAISS)']]
print(f"\naverage chunks shared by Classic and GraphRAG (top-{K}): {sum(both) / max(len(both), 1):.2f}")

## 4. Vector-only vs hybrid, end to end

In [ ]:
rows_vector, summary_vector = evaluate(top_k=5, use_graph=False, tag="vector", verbose=False)

def pct(rate):
    ok, total = rate
    return f"{100 * ok / total:5.1f}%" if total else "   -  "

print(f"{'metric':14} {'vector only':>12} {'hybrid':>8}")
for metric in ("intent", "framework", "review_ok", "cited"):
    print(f"{metric:14} {pct(summary_vector['ALL'][metric]):>12} {pct(summary_hybrid['ALL'][metric]):>8}")
print(f"{'avg latency s':14} {summary_vector['ALL']['latency_s']:>12} {summary_hybrid['ALL']['latency_s']:>8}")

## 5. Chart

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed (pip install matplotlib): the tables above hold the same numbers.")
else:
    cats = [c for c in summary_hybrid if c != "ALL"]
    def share(summary, metric):
        out = []
        for c in cats:
            ok, total = summary[c][metric]
            out.append(100 * ok / total if total else 0)
        return out

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, metric, title in zip(axes, ("intent", "review_ok"), ("Intent accuracy (%)", "Answers accepted by the reviewer (%)")):
        x = range(len(cats))
        ax.bar([i - 0.2 for i in x], share(summary_vector, metric), width=0.4, label="vector only", color="#9db7e0")
        ax.bar([i + 0.2 for i in x], share(summary_hybrid, metric), width=0.4, label="hybrid", color="#0b2545")
        ax.set_xticks(list(x)); ax.set_xticklabels(cats, rotation=30, ha="right"); ax.set_title(title)
    axes[0].legend(); plt.tight_layout(); plt.show()